# Data Integrity and EDA

**Purpose:** Sprint 1 first pass over Meridian Savings data. Validate the CSVs, build a customer × month table with **true running balance**, explore churn thresholds, then **lock** a defensible `Is_Churned` label. §9 closes the post-Sam red-team checklist (P-02/P-03/P-07/P-09).

**Sam decisions (2026-07-10):**
- We own balance / inactivity thresholds — defend with data to Chloe, Jamie, and Sam
- Sense-check balances; round to the nearest currency unit (Sam said “pound”; data is `$`)
- Predict churn in the **next month** (1-month forward window)
- Grace period for new accounts after `creation_date`
- Null `start_balance` (3 rows): **`fillna(0)` confirmed**

**Locked label (see §8):** at observation month `t`, `Is_Churned = 1` iff month `t+1` has `Running_Balance_Rounded == 0` **and** zero active (nonzero) transactions, with tenure ≥ 2 months at `t`.

| | |
|--|--|
| **Inputs** | `StartingFiles/customers_tm1_e.csv`, `StartingFiles/transactions_tm1_e.csv` |
| **In-memory outputs** | `monthly_tx` with `gone_this_month`, `Is_Churned`, `Is_Train_Eligible` |
| **Docs** | `Docs/Sprint1/churn_definition.md`, ADR 004, `Docs/Sprint1/ML_Learnings.md` |
| **Column names** | snake_case only — `.cursor/skills/validate-csv-schema/schema.json` |

**How to read this notebook:** markdown cells state *why* and I/O; code comments mark decisions a junior would not infer from pandas syntax. Do not change the locked label without updating ADR 004 in the same change.


In [ ]:
# JUNIOR_DOC: setup
# Core libraries for EDA. DATA_DIR is cwd-independent (notebook lives under Code/Sprint1/).
# GRACE_MONTHS is locked with the label in §8 — change only with ADR 004.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

# Notebook lives in Code/Sprint1/ — resolve CSVs from repo root regardless of kernel cwd.
DATA_DIR = Path('../..') / 'StartingFiles'
assert (DATA_DIR / 'customers_tm1_e.csv').exists(), f'Missing customers CSV under {DATA_DIR.resolve()}'
assert (DATA_DIR / 'transactions_tm1_e.csv').exists(), f'Missing transactions CSV under {DATA_DIR.resolve()}'

# Locked modeling constants (defended in §5–§8).
GRACE_MONTHS = 2  # exclude observations with Months_Since_Creation < GRACE_MONTHS


## 1. Load Data

**Purpose:** Bring both raw extracts into memory and sanity-check row counts.

**In → Out:** CSV paths under `DATA_DIR` → `customers_df`, `transactions_df`.

**Gotchas:**
- Transactions are large (~350 MB on disk, ~1 GB in RAM). Prefer summaries over dumping full frames into chat/logs.
- Use `transaction_date` (not a Title Case invent). Schema: `validate-csv-schema`.


In [ ]:
# JUNIOR_DOC: load
# Load both extracts. customers = static attributes; transactions = cash movements (~6.4M rows).
# Static customer attributes (state, date of birth, opening balance, etc.)
customers_df = pd.read_csv(DATA_DIR / 'customers_tm1_e.csv')

# All individual transactions — the largest table in the project.
transactions_df = pd.read_csv(DATA_DIR / 'transactions_tm1_e.csv')

display(customers_df.head())
display(transactions_df.head())
print(f'customers={len(customers_df):,}  transactions={len(transactions_df):,}')


## 2. Inspect Schema & Missing Values

**Purpose:** Confirm dtypes/nulls before any aggregation — especially `start_balance` (only known nulls on customers).

**In → Out:** raw frames → printed `.info()` / null counts (no mutation yet).

**Gotchas:**
- Exactly **3** null `start_balance` rows → Sam: `fillna(0)` later in §4 (not here).
- **§2.1 (P-08 / P-10):** integrity checklist + account cardinality. Informational only — does **not** change `Is_Churned`.


In [ ]:
# JUNIOR_DOC: schema peek — no mutation; Sam fillna(0) happens in §4.
print('Customers Info:')
customers_df.info()
print('\nMissing Values:\n', customers_df.isnull().sum())

print('\nTransactions Info:')
transactions_df.info()
print('\nMissing Values:\n', transactions_df.isnull().sum())

# Known DQ: 3 null start_balance values.
# Stakeholder decision (Sam): impute with fillna(0) in Section 4.


### 2.1 Integrity checklist (P-08) & account cardinality (P-10)

Pass/fail DQ beyond nulls. Rules (from Data Dictionary / red-team):

| Check | Pass rule |
|-------|-----------|
| Dup `customer_id` | 0 duplicates |
| Exact dup transaction rows | 0 full-row duplicates |
| Orphan `customer_id` in tx | every tx customer exists in customers |
| Impossible `dob` | parseable; `dob < creation_date`; age at open ≥ 18; age as-of 2020-05-31 ≤ 100; not after as-of |
| Bad `state` | US state/DC **full name** (data is not 2-letter codes) |
| `date` vs `transaction_date` | same calendar month; `transaction_date <= date` (month-end bucket) |
| Amount identity | `amount == deposit + withdrawal` (withdrawal is signed negative) |
| Component signs | `deposit >= 0`, `withdrawal <= 0` |

**P-10:** report share of customers with exactly one `account_id` vs multi-account (pipeline aggregates by `customer_id` only — multi-account would blend balances).

### FAIL triage (first run → after `dob` fix)

| Finding | Count | Disposition |
|---------|-------|-------------|
| Exact dup tx rows | 62 extras (~6.4M rows) | **Note in DQ** — real duplicate withdrawals, not `$0` placeholders. Negligible; optional `drop_duplicates()` before monthly agg. Does **not** unlock label. |
| `dob` “under 18” | 6,722 → **3,007** + 1 over-100 | `days/365.25` caused ~3.7k false FAILs. Remaining 3,007 are under-18 at open by completed years (~2.1%). Note in DQ; do not drop for label. |
| Bad `state` | 21 | **Note in DQ** — mix of codes (`NY`,`TX`,`MASS`,`CALIFORNIA`), junk (`-999`,`UNK`), and `Australia`. Tiny; document; map aliases later for P-09 if needed. |
| P-10 cardinality | 100% 1:1 | Closed — safe to aggregate by `customer_id` only. |
| P-01 mid-span gaps | 0 missing; 100% `shift(-1)` align | Closed — no continuous-month reindex required (placeholders close the panel). |

Full DQ write-up: `Docs/Data/Data_Dictionary.md` §1.3.


In [ ]:
# JUNIOR_DOC P-08/P-10: DQ only — PASS/FAIL table; never mutates frames or Is_Churned.
# Age uses completed calendar years (not days/365.25) to avoid false under-18 FAILs.
# P-08 / P-10 — integrity checklist. Informational only; does not mutate frames or the label.
AS_OF = pd.Timestamp('2020-05-31')

US_STATE_NAMES = {
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',
    'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia',
    'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky',
    'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
    'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
    'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota',
    'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island',
    'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
    'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming',
}

dob = pd.to_datetime(customers_df['dob'], errors='coerce')
creation = pd.to_datetime(customers_df['creation_date'], errors='coerce')

# Completed calendar years (avoids days/365.25 false FAILs at the 18th birthday).
def _completed_years(born: pd.Series, on) -> pd.Series:
    if isinstance(on, pd.Timestamp):
        years = on.year - born.dt.year
        before_bday = (born.dt.month > on.month) | (
            (born.dt.month == on.month) & (born.dt.day > on.day)
        )
    else:
        years = on.dt.year - born.dt.year
        before_bday = (born.dt.month > on.dt.month) | (
            (born.dt.month == on.dt.month) & (born.dt.day > on.dt.day)
        )
    return years - before_bday.astype(int)

age_at_open = _completed_years(dob, creation)
age_as_of = _completed_years(dob, AS_OF)
impossible_dob = (
    dob.isna()
    | creation.isna()
    | (dob > AS_OF)
    | (dob >= creation)
    | (age_at_open < 18)
    | (age_as_of > 100)
)

tx_date = pd.to_datetime(transactions_df['date'], errors='coerce')
tx_event = pd.to_datetime(transactions_df['transaction_date'], errors='coerce')
month_mismatch = tx_date.dt.to_period('M') != tx_event.dt.to_period('M')
event_after_bucket = tx_event > tx_date

# Float-safe identity: amount should equal deposit + withdrawal (withdrawal already negative).
amount_mismatch = ~np.isclose(
    transactions_df['amount'].to_numpy(dtype=float),
    (transactions_df['deposit'] + transactions_df['withdrawal']).to_numpy(dtype=float),
    rtol=0.0,
    atol=1e-6,
    equal_nan=False,
)
bad_signs = (transactions_df['deposit'] < 0) | (transactions_df['withdrawal'] > 0)

orphan_cust = ~transactions_df['customer_id'].isin(customers_df['customer_id'])

# P-10: accounts per customer (and reverse: customers per account).
accts_per_customer = transactions_df.groupby('customer_id')['account_id'].nunique()
custs_per_account = transactions_df.groupby('account_id')['customer_id'].nunique()
n_customers_in_tx = accts_per_customer.shape[0]
n_multi_account = int((accts_per_customer > 1).sum())
n_shared_account = int((custs_per_account > 1).sum())
one_to_one_rate = float((accts_per_customer == 1).mean()) if n_customers_in_tx else float('nan')

def _row(check_id, description, fail_count, notes=''):
    return {
        'check_id': check_id,
        'description': description,
        'fail_count': int(fail_count),
        'status': 'PASS' if fail_count == 0 else 'FAIL',
        'notes': notes,
    }

integrity_rows = [
    _row('P08-dup-customer', 'Duplicate customer_id in customers', customers_df['customer_id'].duplicated().sum()),
    _row('P08-dup-tx-row', 'Exact duplicate transaction rows', transactions_df.duplicated().sum()),
    _row('P08-orphan-customer', 'Tx customer_id missing from customers', orphan_cust.sum()),
    _row(
        'P08-dob',
        'Impossible dob (parse / future / vs creation / age)',
        impossible_dob.sum(),
        'completed years: age_at_open>=18; age_as_of<=100; dob<creation; dob<=2020-05-31',
    ),
    _row('P08-state', 'state not in US full-name set (+ DC)', (~customers_df['state'].isin(US_STATE_NAMES)).sum()),
    _row(
        'P08-date-month',
        'date month != transaction_date month',
        month_mismatch.sum(),
        'date=month-end bucket; transaction_date=event',
    ),
    _row('P08-date-order', 'transaction_date > date (after month-end)', event_after_bucket.sum()),
    _row('P08-amount-id', 'amount != deposit + withdrawal', amount_mismatch.sum(), 'atol=1e-6'),
    _row('P08-signs', 'deposit<0 or withdrawal>0', bad_signs.sum()),
]

integrity_df = pd.DataFrame(integrity_rows)
print('=== P-08 Integrity checklist ===')
display(integrity_df)

print('\n=== P-10 account_id : customer_id cardinality ===')
print(f'Customers appearing in transactions: {n_customers_in_tx:,}')
print(f'Distinct account_id: {transactions_df["account_id"].nunique():,}')
print(f'Customers with exactly 1 account: {(accts_per_customer == 1).sum():,} ({one_to_one_rate:.2%})')
print(f'Customers with >1 account: {n_multi_account:,} ({n_multi_account / n_customers_in_tx:.2%})')
print(f'Accounts linked to >1 customer: {n_shared_account:,}')
if n_multi_account == 0 and n_shared_account == 0:
    print('Cardinality: 1:1 (safe to aggregate by customer_id only).')
else:
    print('WARNING: multi-account or shared-account rows exist — customer-level balances may blend accounts (RT-10).')

p10_summary = pd.DataFrame(
    [
        {
            'check_id': 'P10-cardinality',
            'one_to_one_customer_rate': one_to_one_rate,
            'multi_account_customers': n_multi_account,
            'shared_account_ids': n_shared_account,
            'status': 'DOCUMENTED',
        }
    ]
)
display(p10_summary)

# --- FAIL triage detail (for DQ notes; does not mutate frames) ---
print('\n=== FAIL triage detail ===')
bad_states = customers_df.loc[~customers_df['state'].isin(US_STATE_NAMES), 'state']
if len(bad_states):
    print('Bad state value_counts:')
    print(bad_states.value_counts().to_string())
else:
    print('Bad state: none')

n_dup_extra = int(transactions_df.duplicated().sum())
print(f'Exact duplicate tx extras: {n_dup_extra}')
if n_dup_extra:
    print('(Disposition: note in DQ; optional drop_duplicates before monthly agg — label lock unchanged.)')

if int(impossible_dob.sum()):
    print('Remaining impossible_dob breakdown:')
    print(
        pd.DataFrame(
            {
                'parse_fail': dob.isna() | creation.isna(),
                'future_dob': dob > AS_OF,
                'dob_ge_creation': dob >= creation,
                'under_18_at_open': age_at_open < 18,
                'over_100_asof': age_as_of > 100,
            }
        )
        .loc[impossible_dob]
        .sum()
        .to_string()
    )
else:
    print('Impossible dob: none after completed-year age check.')


## 3. Zero-Amount Placeholders (Critical for Inactivity)

**Purpose:** Detect month-end `$0` padding rows so inactivity is not measured with raw row counts.

**In → Out:** `transactions_df` → adds `is_active_tx` (True = real cash movement).

**Why it matters:** Rows with `amount == deposit == withdrawal == 0` pad quiet months. A raw `groupby(...).size()` is almost never zero → “0 transactions” is a **false** inactivity test.

**Rule:** activity = nonzero cash movement (`is_active_tx`). Expect ~8% placeholders.


In [ ]:
# JUNIOR_DOC: placeholders — is_active_tx drives inactivity forever after this cell.
# $0 rows pad quiet months; without this flag, Monthly_Transaction_Count never hits 0.
zero_mask = (
    (transactions_df['amount'] == 0)
    & (transactions_df['deposit'] == 0)
    & (transactions_df['withdrawal'] == 0)
)
print(
    f'$0 placeholder rows: {zero_mask.sum():,} / {len(transactions_df):,} '
    f'({100 * zero_mask.mean():.2f}%)'
)

transactions_df['is_active_tx'] = ~zero_mask
display(
    transactions_df.loc[
        zero_mask, ['transaction_date', 'amount', 'deposit', 'withdrawal']
    ].head()
)


## 4. Aggregate to Customer × Month & Running Balance

**Purpose:** Collapse daily txs to a customer × calendar-month panel and reconstruct account state.

**In → Out:** `transactions_df` + customer opening fields → `monthly_tx` with:
`Monthly_Active_Txn_Count`, `Monthly_Amount_Sum`, `Running_Balance`, `Running_Balance_Rounded`, `Months_Since_Creation`, `Is_Inactive_Month`.

**Key decisions:**
- `Running_Balance = start_balance + cumsum(monthly net amount)` — must **sort** by customer/month before `cumsum`.
- Null `start_balance` → `fillna(0)` (**Sam-confirmed**).
- Round to nearest `$1` for Sam’s sense-check (`Running_Balance_Rounded`).

**P-01 / RT-01:** After the rollup, §4.1 measures missing calendar months. If gaps are material, reindex (active=0, carry balance). Confirm with the gap check before changing the panel — placeholders may already close silent months.


In [ ]:
# JUNIOR_DOC: build monthly_tx
# Steps: period month → monthly agg (active count + net amount) → merge opening balance
# → sort → running balance via cumsum → tenure → inactive flag.
# Why sort before cumsum: otherwise Running_Balance is wrong within a customer.
# --- Step 1: Parse dates and bucket into calendar months ---
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])
# Collapse each timestamp to its calendar month (e.g. 2007-01-15 → 2007-01).
transactions_df['Month'] = transactions_df['transaction_date'].dt.to_period('M')

customers_df['creation_date'] = pd.to_datetime(customers_df['creation_date'])

null_start = customers_df[customers_df['start_balance'].isna()]
print('Customers with null start_balance (will fillna(0) per Sam):')
display(null_start)

# --- Step 2: Monthly rollup ---
monthly_tx = transactions_df.groupby(['customer_id', 'Month'], as_index=False).agg(
    Monthly_Transaction_Count=('amount', 'size'),
    Monthly_Active_Txn_Count=('is_active_tx', 'sum'),
    Monthly_Amount_Sum=('amount', 'sum'),
)

# --- Step 3: Opening balance + account open date ---
cust_cols = customers_df[['customer_id', 'start_balance', 'creation_date', 'state']].copy()
# Stakeholder decision (Sam, Head of DS): treat null opening balances as $0.
cust_cols['start_balance'] = cust_cols['start_balance'].fillna(0)
monthly_tx = monthly_tx.merge(cust_cols, on='customer_id', how='left')

# --- Step 4: True running balance ---
monthly_tx = monthly_tx.sort_values(['customer_id', 'Month']).reset_index(drop=True)
monthly_tx['Running_Balance'] = (
    monthly_tx['start_balance']
    + monthly_tx.groupby('customer_id')['Monthly_Amount_Sum'].cumsum()
)
monthly_tx['Running_Balance_Rounded'] = monthly_tx['Running_Balance'].round(0)

month_end = monthly_tx['Month'].dt.to_timestamp(how='end')
monthly_tx['Months_Since_Creation'] = (
    (month_end.dt.year - monthly_tx['creation_date'].dt.year) * 12
    + (month_end.dt.month - monthly_tx['creation_date'].dt.month)
)

monthly_tx['Is_Inactive_Month'] = monthly_tx['Monthly_Active_Txn_Count'] == 0

display(monthly_tx.head(12))
n_inactive = int(monthly_tx['Is_Inactive_Month'].sum())
print(f'Inactive months (active_txn_count==0): {n_inactive:,} / {len(monthly_tx):,}')


### 4.1 Panel continuity (P-01)

**Purpose:** Prove that `shift(-1)` is next **calendar** month, not merely next *observed* row (RT-01).

**In → Out:** `monthly_tx` (read-only) → gap metrics; does **not** mutate the panel unless we later decide to reindex.

| Metric | Meaning |
|--------|---------|
| Missing months | `expected_span_months - n_observed` per customer |
| `shift(-1)` alignment | Share of rows where next observed month == next calendar month |

**Decision rule:** if gap rate is material → reindex (fill active/amount 0, carry `Running_Balance`). If gap rate ≈ 0 → keep observed panel. Label lock unchanged unless a reindex changes who is labeled (then update ADR 004).

**Outcome (run):** 0% gaps; 100% alignment → no reindex.


In [ ]:
# JUNIOR_DOC P-01: if gap_rate≈0, do not reindex — placeholders already close the panel.
# P-01 — calendar-month gap check on the observed customer × month panel.
# Does not mutate monthly_tx unless we later decide gaps are material.
panel = monthly_tx.sort_values(['customer_id', 'Month']).copy()
g = panel.groupby('customer_id')['Month']
first_m = g.transform('min')
last_m = g.transform('max')
n_obs = g.transform('size')
# Inclusive month count in [first, last] via period arithmetic.
n_expected = (last_m - first_m).apply(lambda d: d.n + 1)
# One row per customer for gap totals
cust_gap = (
    pd.DataFrame(
        {
            'customer_id': panel['customer_id'],
            'n_obs': n_obs,
            'n_expected': n_expected,
        }
    )
    .drop_duplicates('customer_id')
    .assign(n_missing=lambda d: d['n_expected'] - d['n_obs'])
)

missing_sum = int(cust_gap['n_missing'].sum())
customers_with_gap = int((cust_gap['n_missing'] > 0).sum())
gap_rate = float(cust_gap['n_missing'].sum() / cust_gap['n_expected'].sum())

panel['next_obs_month'] = panel.groupby('customer_id')['Month'].shift(-1)
panel['next_cal_month'] = panel['Month'] + 1
has_next = panel['next_obs_month'].notna()
aligned = panel.loc[has_next, 'next_obs_month'] == panel.loc[has_next, 'next_cal_month']
align_rate = float(aligned.mean()) if has_next.any() else float('nan')
misaligned = int((~aligned).sum()) if has_next.any() else 0

print('=== P-01 panel continuity ===')
print(f'Customer-months observed: {len(panel):,}')
print(f'Customers: {cust_gap.shape[0]:,}')
print(f'Missing calendar months (sum): {missing_sum:,}')
print(f'Customers with any mid-span gap: {customers_with_gap:,}')
print(f'Gap rate (missing / expected span months): {gap_rate:.4%}')
print(f"shift(-1) == next calendar month: {align_rate:.4%} ({misaligned:,} misaligned)")

p01_summary = pd.DataFrame(
    [
        {
            'check_id': 'P01-gaps',
            'missing_months': missing_sum,
            'customers_with_gap': customers_with_gap,
            'gap_rate': gap_rate,
            'shift_next_calendar_align_rate': align_rate,
            'reindex_required': gap_rate > 0,
            'status': 'PASS_NO_REINDEX' if gap_rate == 0 and align_rate == 1.0 else 'ACTION_NEEDED',
        }
    ]
)
display(p01_summary)

if gap_rate == 0 and align_rate == 1.0:
    print(
        'Decision: no continuous-month reindex. '
        '$0 placeholders (or continuous activity) already close the panel; '
        'next-month labels are calendar-aligned. Label lock unchanged.'
    )
else:
    print(
        'Decision: gaps are material — implement reindex '
        '(active=0, amount=0, carry Running_Balance) before trusting shift(-1) labels.'
    )


## 5. Churn Definition EDA (Post-Sam)

**Purpose:** Defend balance / inactivity / grace / horizon choices with data before locking §8.

**Anti-leakage sketch:** features from month `t` only; label from month `t+1`.

**Subsections (code cells below):**
| Cell | Question |
|------|----------|
| **5a** | Which balance rule matches “near $0”? |
| **5b** | How long are inactive / near-zero streaks? High-balance dormancy? |
| **5c** | How many months of new-account grace? |
| **5d** | 1-month vs 2-month confirmation (sensitivity only) |
| **5e** | Exit shape + coarse AUM proxy (true AUM is §9 / P-02) |

**Offline full-file facts (also reproduced here):**
- ~8.15% `$0` placeholders; ~266k months rounded-$0 **and** inactive
- Most near-zero inactive streaks length **1** (~163k) vs length 2 (~34k)
- ~53k high-balance dormant streaks (min ≥ $1000) — **not** churn
- Next-month positive rate ≈ **11.8%** (1m) vs **2.6%** (2m confirm)


### 5a. Balance threshold sweep

**Purpose:** Compare candidate “near $0” rules and pick the one that matches Sam’s sense-check.

**In → Out:** `monthly_tx` → summary table + histogram of balances with `|balance| ≤ $50`.

**Why `rounded == 0`:** Empirically ≈ `|balance| < 1`; looser rules (`≤ $50`) pull in non-abandoned low balances and large negatives.


In [ ]:
# JUNIOR_DOC 5a: compare near-$0 rules; locked choice is rounded_eq_0 (see §7).
# --- 5a. Balance threshold sweep ---
thresholds = {
    'rounded_eq_0': monthly_tx['Running_Balance_Rounded'] == 0,
    'abs_lt_1': monthly_tx['Running_Balance'].abs() < 1,
    'abs_lt_5': monthly_tx['Running_Balance'].abs() < 5,
    'abs_le_50': monthly_tx['Running_Balance'].abs() <= 50,
    'le_50_signed': monthly_tx['Running_Balance'] <= 50,
}

rows = []
for name, mask in thresholds.items():
    rows.append({
        'rule': name,
        'customer_months': int(mask.sum()),
        'and_inactive': int((mask & monthly_tx['Is_Inactive_Month']).sum()),
        'pct_of_all_months': round(100 * mask.mean(), 2),
    })
threshold_summary = pd.DataFrame(rows)
display(threshold_summary)

near_zero = monthly_tx[monthly_tx['Running_Balance'].abs() <= 50]
plt.figure(figsize=(10, 5))
sns.histplot(near_zero['Running_Balance'], bins=50)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Running balances with |balance| <= $50')
plt.xlabel('Running Balance ($)')
plt.ylabel('Customer-months')
plt.show()


### 5b. Consecutive inactivity streaks

**Purpose:** See whether “gone” looks like a short blip or long dormancy; separate high-balance quiet accounts.

**In → Out:** streak length tables; flags for streaks ending near $0 vs min balance ≥ $1000.

**Why it matters:** Mostly length-1 near-zero streaks → do **not** require 2-month confirmation for the training label. High-balance dormancy preserves AUM → **not** churn.


In [ ]:
# JUNIOR_DOC 5b: streak_id flips when inactive toggles; high_balance_dormant ≠ churn.
# --- 5b. Consecutive inactivity streaks ---


def streak_id(inactive: pd.Series) -> pd.Series:
    return inactive.ne(inactive.shift(fill_value=False)).cumsum()


monthly_tx['inactive_streak_id'] = (
    monthly_tx.groupby('customer_id', group_keys=False)['Is_Inactive_Month']
    .transform(streak_id)
)

inactive_only = monthly_tx[monthly_tx['Is_Inactive_Month']].copy()
streaks = (
    inactive_only.groupby(['customer_id', 'inactive_streak_id'], as_index=False)
    .agg(
        streak_len=('Month', 'size'),
        min_balance=('Running_Balance', 'min'),
        end_balance_rounded=('Running_Balance_Rounded', 'last'),
    )
)

print('Inactive streak length distribution:')
display(streaks['streak_len'].value_counts().sort_index().head(12).to_frame('n_streaks'))

streaks['ends_near_zero'] = streaks['end_balance_rounded'].abs() <= 0
streaks['high_balance_dormant'] = streaks['min_balance'] >= 1000

print('\nStreaks ending at rounded £0, by length:')
display(
    streaks.loc[streaks['ends_near_zero'], 'streak_len']
    .value_counts()
    .sort_index()
    .head(12)
    .to_frame('n_streaks')
)

print('\nHigh-balance dormant streaks (min balance >= $1000):')
display(
    streaks.loc[streaks['high_balance_dormant'], 'streak_len']
    .describe()
    .to_frame('high_bal_dormant')
)


### 5c. New-account grace period

**Purpose:** Quantify how many customer-months a 1 / 2 / 3 month grace would exclude.

**In → Out:** table + tenure histogram. Locked choice: `GRACE_MONTHS = 2` (Sam).


In [ ]:
# JUNIOR_DOC 5c: compare grace 1/2/3; production uses GRACE_MONTHS=2 from cell 1.
# --- 5c. New-account grace period ---
grace_rows = []
for g in [1, 2, 3]:
    in_grace = monthly_tx['Months_Since_Creation'] < g
    grace_rows.append({
        'grace_months': g,
        'customer_months_excluded': int(in_grace.sum()),
        'pct_months_excluded': round(100 * in_grace.mean(), 2),
        'distinct_customers_touched': int(
            monthly_tx.loc[in_grace, 'customer_id'].nunique()
        ),
    })
display(pd.DataFrame(grace_rows))

plt.figure(figsize=(10, 4))
sns.histplot(monthly_tx['Months_Since_Creation'].clip(upper=60), bins=60)
plt.title('Months since creation (clipped at 60)')
plt.xlabel('Months since creation')
plt.show()


### 5d. One-month vs two-month confirmation (sensitivity)

**Purpose:** Compare next-month positive rates if “gone” must hold for one month vs two consecutive months.

**In → Out:** adds `Churn_State_This_Month`, `Churn_State_2m_Confirm` (sensitivity columns — **not** the training label).

**Expect:** ~11.8% (1m) vs ~2.6% (2m). Training uses the 1m rule in §8.


In [ ]:
# JUNIOR_DOC 5d: sensitivity only — training label is built in §8 from the 1m rule.
# shift(-1) = next observed month; P-01 showed that equals next calendar month here.
# --- 5d. Compare 1-month vs 2-month confirmation (sensitivity) ---
# Candidate state *in this month*: near-zero rounded + inactive.
monthly_tx['Churn_State_This_Month'] = (
    (monthly_tx['Running_Balance_Rounded'] == 0)
    & monthly_tx['Is_Inactive_Month']
).astype(int)

prev_state = monthly_tx.groupby('customer_id')['Churn_State_This_Month'].shift(1)
monthly_tx['Churn_State_2m_Confirm'] = (
    (monthly_tx['Churn_State_This_Month'] == 1) & (prev_state == 1)
).astype(int)

eligible = monthly_tx['Months_Since_Creation'] >= GRACE_MONTHS
for col in ['Churn_State_This_Month', 'Churn_State_2m_Confirm']:
    y_next = monthly_tx.groupby('customer_id')[col].shift(-1)
    known = monthly_tx.loc[eligible].copy()
    known['_y'] = y_next.loc[eligible]
    known = known[known['_y'].notna()]
    print(
        f'{col}: known_rows={len(known):,}  '
        f'next_month_rate={known["_y"].mean():.4%}'
    )


### 5e. AUM / exit shape (coarse proxy)

**Purpose:** Visualize balances in left-looking months; print the old coarse AUM proxy for continuity.

**Gotcha:** Summing last-month `start_balance` (~$2.2M) is **not** true AUM at exit. **P-02 in §9** computes prior-3-month max (~$7.27M) for the PRD.


In [ ]:
# JUNIOR_DOC 5e: coarse start_balance proxy kept for continuity; true AUM is §9 P-02.
# --- 5e. AUM / exit shape ---
# Coarse proxy kept for continuity; **P-02 true AUM-at-exit** is computed in §9
# (prior-month Running_Balance at first left-looking transition).
exit_months = monthly_tx[monthly_tx['Churn_State_This_Month'] == 1]
print(f'Near-zero+inactive customer-months: {len(exit_months):,}')
print(f'Distinct customers ever in that state: {exit_months["customer_id"].nunique():,}')

plt.figure(figsize=(10, 4))
sns.histplot(exit_months['Running_Balance'].clip(-50, 50), bins=40)
plt.title('Balance at near-zero+inactive months (clipped [-50, 50])')
plt.xlabel('Running Balance ($)')
plt.show()

last_month = monthly_tx.groupby('customer_id', as_index=False).tail(1)
final_exit = last_month[last_month['Churn_State_This_Month'] == 1]
print(f'Last-month exits (1m rule): {len(final_exit):,}')
print(f'Sum start_balance (coarse AUM proxy): ${final_exit["start_balance"].sum():,.0f}')
print('→ Run §9 (P-02) for true AUM-at-exit before updating the PRD $4.2M placeholder.')


## 6. Visualizations

**Purpose:** Quick distribution plots that support the locked thresholds (balance + activity).

**In → Out:** `monthly_tx` → figures only (no new columns).

**Note:** Eligible volume / positive rate **by Month (P-11)** is plotted in §8.1, not here.


In [ ]:
# JUNIOR_DOC §6: distribution support for locked thresholds (clipped for readability).
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(monthly_tx['Running_Balance'].clip(-500, 5000), bins=50, ax=axes[0])
axes[0].set_title('Running balance (clipped [-500, 5000])')
axes[0].set_xlabel('Running Balance ($)')

sns.histplot(
    monthly_tx.loc[monthly_tx['Monthly_Active_Txn_Count'] <= 20, 'Monthly_Active_Txn_Count'],
    bins=21,
    discrete=True,
    ax=axes[1],
)
axes[1].set_title('Active (nonzero) transactions per month')
axes[1].set_xlabel('Monthly_Active_Txn_Count')

plt.tight_layout()
plt.show()


## 7. Decision Rationale (Locked)

**Purpose:** One-page “why this label” for Sam / Chloe / Jamie — maps EDA evidence to ADR 004.

| Choice | Decision | Why |
|--------|----------|-----|
| Balance | `Running_Balance_Rounded == 0` | Matches Sam (“near $0”, round to nearest unit); ≈ `abs < 1` |
| Activity | `Monthly_Active_Txn_Count == 0` | Ignores `$0` placeholders; raw count is invalid |
| Horizon | Label = state in **t+1** | 1-month forward prediction; features stop at end of `t` |
| Grace | Tenure ≥ 2 at `t` | Sam-approved new-account exclusion |
| High-balance dormant | **Not** churn | Inactive with balance ≥ $1000 preserves AUM |
| 2-month confirm | Sensitivity only | Most streaks length 1; 2m drops rate to ~2.6% and delays detection |

Formal ADR: `Docs/Architecture/ADRs/004-churn-target-and-predictive-window.md`.


## 8. Locked `Is_Churned` Definition

**Purpose:** Materialize the training target on `monthly_tx` exactly as ADR 004 states.

**In → Out:** adds `gone_this_month`, `Is_Churned` (next-month state), `Is_Train_Eligible` (tenure ≥ `GRACE_MONTHS` and known next month).

**Gotchas:**
- Last panel month → `Is_Churned` is NaN (right-censoring) — correctly excluded from training.
- Do **not** use month `t+1` cash-flow as a feature when predicting `Is_Churned` at `t`.
- Changing this math requires ADR 004 + `churn_definition.md` in the same change.


In [ ]:
# JUNIOR_DOC §8: ADR 004 materialization.
# gone_this_month = left-looking *this* month; Is_Churned = left-looking *next* month.
# Is_Train_Eligible drops grace + right-censored last month (NaN label).
# --- Locked training label: churn in the *next* month ---
# State that defines "gone" in a calendar month:
monthly_tx['gone_this_month'] = (
    (monthly_tx['Running_Balance_Rounded'] == 0)
    & (monthly_tx['Monthly_Active_Txn_Count'] == 0)
).astype(int)

# Label for row at month t = whether month t+1 is "gone".
monthly_tx['Is_Churned'] = (
    monthly_tx.groupby('customer_id')['gone_this_month'].shift(-1)
)

# Training-eligible rows: past grace period and known next month.
monthly_tx['Is_Train_Eligible'] = (
    (monthly_tx['Months_Since_Creation'] >= GRACE_MONTHS)
    & monthly_tx['Is_Churned'].notna()
)

eligible = monthly_tx[monthly_tx['Is_Train_Eligible']]
print('Locked label summary (tenure >= 2, known next month):')
print(f'  rows={len(eligible):,}')
print(f'  positive rate={eligible["Is_Churned"].mean():.4%}')
print(f'  positives={int(eligible["Is_Churned"].sum()):,}')
print(f'  negatives={int((eligible["Is_Churned"] == 0).sum()):,}')

display(
    monthly_tx.loc[
        monthly_tx['Is_Churned'] == 1,
        [
            'customer_id',
            'Month',
            'Running_Balance',
            'Monthly_Active_Txn_Count',
            'gone_this_month',
            'Is_Churned',
            'Months_Since_Creation',
        ],
    ].head(12)
)

print('\nFormal definition locked. Use Is_Churned with Is_Train_Eligible for modeling.')


### 8.1 Label interpretability (P-04 / P-05 / P-11)

**Purpose:** Unpack “11.8% churn” so stakeholders do not hear a unique-customer forever-churn rate.

| ID | Question | What we report |
|----|----------|----------------|
| **P-04** | Prevalence vs first exit vs reactivation | Table on eligible rows |
| **P-05** | Are we predicting continued emptiness? | Metrics on `gone_this_month == 0` (impending) + majority-class baseline |
| **P-11** | Time trend + censoring | Eligible volume / positive rate by `Month`; last month excluded by design |

**Expected ballpark (full run):** prevalence ~11.76%; first-exit ~3.92% of rows; ~64% customers ever positive; impending ~10.4% vs already-gone ~22.2%; late panel ~12–14%.


In [ ]:
# JUNIOR_DOC P-04/05/11: unpack 11.8% — prevalence ≠ unique-customer churn; impending cohort baseline.
# P-04 / P-05 / P-11 — interpretability on the locked label (read-only cuts).
eligible = monthly_tx.loc[monthly_tx['Is_Train_Eligible']].copy()
eligible = eligible.sort_values(['customer_id', 'Month']).reset_index(drop=True)

# --- P-04: prevalence vs first-exit vs reactivations ---
n_elig_rows = len(eligible)
n_pos_rows = int(eligible['Is_Churned'].sum())
prevalence = float(eligible['Is_Churned'].mean())
n_elig_customers = eligible['customer_id'].nunique()
n_customers_ever_pos = eligible.loc[eligible['Is_Churned'] == 1, 'customer_id'].nunique()

pos = eligible['Is_Churned'] == 1
pos_order = pos.groupby(eligible['customer_id']).cumsum()
is_first_exit_row = pos & (pos_order == 1)
n_first_exit = int(is_first_exit_row.sum())
first_exit_row_rate = n_first_exit / n_elig_rows
ever_pos_customer_rate = n_customers_ever_pos / n_elig_customers
share_pos_that_are_first = n_first_exit / n_pos_rows if n_pos_rows else float('nan')
share_pos_already_gone_at_t = float(
    eligible.loc[pos, 'gone_this_month'].mean()
) if n_pos_rows else float('nan')

# Reactivation = left-looking then funded again (panel-wide, not only eligible).
panel = monthly_tx.sort_values(['customer_id', 'Month'])
prev_gone = panel.groupby('customer_id')['gone_this_month'].shift(1)
reactivation = (prev_gone == 1) & (panel['gone_this_month'] == 0)
n_reactivation_events = int(reactivation.sum())
n_customers_reactivated = panel.loc[reactivation, 'customer_id'].nunique()

p04 = pd.DataFrame(
    [
        {
            'metric': 'prevalence_customer_months',
            'value': prevalence,
            'count': n_pos_rows,
            'denom': n_elig_rows,
            'notes': 'Share of eligible rows with Is_Churned=1 (the ~11.8% figure)',
        },
        {
            'metric': 'first_exit_row_rate',
            'value': first_exit_row_rate,
            'count': n_first_exit,
            'denom': n_elig_rows,
            'notes': 'Eligible rows that are a customer\'s first Is_Churned=1',
        },
        {
            'metric': 'customers_ever_positive_rate',
            'value': ever_pos_customer_rate,
            'count': n_customers_ever_pos,
            'denom': n_elig_customers,
            'notes': 'Share of eligible customers with ≥1 positive label',
        },
        {
            'metric': 'share_of_positives_that_are_first_exit',
            'value': share_pos_that_are_first,
            'count': n_first_exit,
            'denom': n_pos_rows,
            'notes': 'Rest are recurring empty-state positives',
        },
        {
            'metric': 'share_of_positives_already_gone_at_t',
            'value': share_pos_already_gone_at_t,
            'count': int(eligible.loc[pos, 'gone_this_month'].sum()) if n_pos_rows else 0,
            'denom': n_pos_rows,
            'notes': 'Already left-looking at observation month t',
        },
        {
            'metric': 'reactivation_events',
            'value': float('nan'),
            'count': n_reactivation_events,
            'denom': int(len(panel)),
            'notes': f'gone→funded transitions; {n_customers_reactivated:,} distinct customers',
        },
    ]
)
print('=== P-04 prevalence vs first-exit vs reactivation ===')
display(p04)

# --- P-05: impending-exit cohort (not already gone at t) ---
impending = eligible.loc[eligible['gone_this_month'] == 0]
n_imp = len(impending)
n_imp_pos = int(impending['Is_Churned'].sum())
imp_rate = float(impending['Is_Churned'].mean()) if n_imp else float('nan')
majority_baseline_acc = 1.0 - imp_rate if n_imp else float('nan')

p05 = pd.DataFrame(
    [
        {
            'cohort': 'all_eligible',
            'rows': n_elig_rows,
            'positive_rate': prevalence,
            'majority_class_accuracy': 1.0 - prevalence,
        },
        {
            'cohort': 'gone_this_month==0 (impending)',
            'rows': n_imp,
            'positive_rate': imp_rate,
            'majority_class_accuracy': majority_baseline_acc,
        },
        {
            'cohort': 'gone_this_month==1 (already gone)',
            'rows': n_elig_rows - n_imp,
            'positive_rate': float(
                eligible.loc[eligible['gone_this_month'] == 1, 'Is_Churned'].mean()
            )
            if (n_elig_rows - n_imp)
            else float('nan'),
            'majority_class_accuracy': float('nan'),
        },
    ]
)
print('\n=== P-05 impending-exit cohort (gone_this_month == 0) ===')
display(p05)
print(
    f'Note: on the impending cohort, predicting all-negative gets '
    f'~{majority_baseline_acc:.2%} accuracy — that is the naive baseline to beat. '
    'Already-gone rows make the full eligible task easier (continued emptiness).'
)

# --- P-11: eligible volume + positive rate by Month ---
by_month = (
    eligible.assign(Month_str=eligible['Month'].astype(str))
    .groupby('Month_str', as_index=False)
    .agg(
        n_eligible=('Is_Churned', 'size'),
        n_positive=('Is_Churned', 'sum'),
        positive_rate=('Is_Churned', 'mean'),
    )
)
# Right-censoring: panel's last calendar month never appears (no t+1).
panel_months = monthly_tx['Month'].astype(str)
last_panel_month = panel_months.max()
print('\n=== P-11 eligible volume / positive rate by Month ===')
print(
    f'Right-censoring: panel last month={last_panel_month} has Is_Churned=NaN '
    f'→ excluded from Is_Train_Eligible (by design).'
)
display(by_month.tail(12))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(by_month['Month_str'], by_month['n_eligible'], color='steelblue')
axes[0].set_ylabel('Eligible rows')
axes[0].set_title('P-11: Eligible volume by Month (tenure≥2, known next month)')
axes[0].tick_params(axis='x', labelbottom=False)

axes[1].plot(by_month['Month_str'], by_month['positive_rate'], color='darkorange')
axes[1].axhline(prevalence, color='gray', ls='--', lw=1, label=f'overall {prevalence:.2%}')
axes[1].set_ylabel('Positive rate')
axes[1].set_xlabel('Month')
axes[1].legend(loc='upper right')
# Sparse x ticks for readability
tick_idx = list(range(0, len(by_month), max(len(by_month) // 12, 1)))
axes[1].set_xticks(tick_idx)
axes[1].set_xticklabels(by_month.loc[tick_idx, 'Month_str'], rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('P-04/P-05/P-11 complete. Label lock unchanged.')


## 9. Revision-plan batch (P-02 / P-03 / P-07 / P-09)

**Purpose:** One runnable block for the remaining *code* red-team items. Run **after** §8 (needs `gone_this_month` / `Is_Churned` / `Is_Train_Eligible`).

| ID | Question | Output |
|----|----------|--------|
| **P-02** | What AUM was at risk before first left-looking exit? | Prior-month balance **and** max balance in prior 3 months |
| **P-03** | When inside the month do drains happen? | Timing in the **prior** month (exit month has 0 active txs by definition) |
| **P-07** | How common are negative balances? | Counts; keep/change note for `rounded == 0` |
| **P-09** | Does churn vary by age / state? | Crosstabs on eligible rows for Jamie |

**Docs-only (already done):** **P-12** currency narrative + **P-06** `ML_Anti_Patterns.md`.

**Gotchas:** Read-only — does **not** change locked `Is_Churned`. P-02 prior-month alone is ~$0 by construction; Morgan/PRD figure uses **prior-3-month max** (~$7.27M). P-03 must not time the exit month.


In [ ]:
# JUNIOR_DOC §9: P-02 prior-3-month max AUM; P-03 time PRIOR month; P-07 keep rounded==0; P-09 Jamie slices.
# P-02 / P-03 / P-07 / P-09 — revision-plan batch (read-only; label lock unchanged).
# Requires: monthly_tx with gone_this_month / Is_Churned / Is_Train_Eligible (§8),
#           transactions_df with is_active_tx (§3), customers_df with dob (§1).

AS_OF = pd.Timestamp('2020-05-31')
panel = monthly_tx.sort_values(['customer_id', 'Month']).copy()

# =============================================================================
# P-02 — AUM before first left-looking transition
# =============================================================================
# First exit = first month with gone_this_month==1.
# Prior-month Running_Balance is often ~$0 (account already drained but still
# "active" the month before). Also report max balance in the prior 3 months —
# that is the Morgan-relevant AUM-at-risk figure.
panel['prev_gone'] = panel.groupby('customer_id')['gone_this_month'].shift(1)
panel['prev_balance'] = panel.groupby('customer_id')['Running_Balance'].shift(1)
panel['bal_l2'] = panel.groupby('customer_id')['Running_Balance'].shift(2)
panel['bal_l3'] = panel.groupby('customer_id')['Running_Balance'].shift(3)

first_exit = panel[
    (panel['gone_this_month'] == 1)
    & (panel['prev_gone'].fillna(0) == 0)
].copy()
first_exit['prior3_max_balance'] = first_exit[['prev_balance', 'bal_l2', 'bal_l3']].max(
    axis=1, skipna=True
)

aum_prior = first_exit['prev_balance'].dropna()
aum_prior3 = first_exit['prior3_max_balance'].dropna()

last_month = panel.groupby('customer_id', as_index=False).tail(1)
final_exit = last_month[last_month['gone_this_month'] == 1]
coarse_aum = float(final_exit['start_balance'].sum())

print('=== P-02 AUM-at-exit (first left-looking transition) ===')
print(f'First-exit events: {len(first_exit):,}')
print(f'Sum prior-month Running_Balance: ${aum_prior.sum():,.0f}  '
      f'(median ${aum_prior.median():,.2f})')
print(f'Sum max(Running_Balance) over prior 3 months: ${aum_prior3.sum():,.0f}  '
      f'(median ${aum_prior3.median():,.2f}; P90 ${aum_prior3.quantile(0.9):,.2f})')
print(f'Coarse §5e proxy (last-month start_balance sum): ${coarse_aum:,.0f}')
print(
    'Interpretation: prior-month alone is near $0 because customers usually '
    'drain before the first inactive+rounded-0 month. Prefer prior-3-month max '
    'for PRD / Morgan AUM-at-risk; do not use start_balance.'
)
display(
    pd.DataFrame(
        {
            'prior_month': aum_prior.describe(percentiles=[0.5, 0.9, 0.99]),
            'prior3_max': aum_prior3.describe(percentiles=[0.5, 0.9, 0.99]),
        }
    )
)

# =============================================================================
# P-03 — Intra-month timing in the PRIOR month (exit month has 0 active txs)
# =============================================================================
exit_keys = first_exit[['customer_id', 'Month', 'prev_balance']].copy()
exit_keys['Prior_Month'] = exit_keys['Month'] - 1

# Opening balance for prior month = balance at end of month-2 (shift 2 from exit)
exit_keys = exit_keys.merge(
    first_exit[['customer_id', 'Month', 'bal_l2']].rename(
        columns={'bal_l2': 'prior_month_open_balance'}
    ),
    on=['customer_id', 'Month'],
    how='left',
)

tx = transactions_df.loc[
    transactions_df['is_active_tx'],
    ['customer_id', 'transaction_date', 'amount'],
].copy()
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])
tx['Month'] = tx['transaction_date'].dt.to_period('M')
tx['day'] = tx['transaction_date'].dt.day

tx_prior = tx.merge(
    exit_keys[['customer_id', 'Prior_Month', 'prior_month_open_balance']].rename(
        columns={'Prior_Month': 'Month'}
    ),
    on=['customer_id', 'Month'],
    how='inner',
)

last_active_prior = (
    tx_prior.groupby(['customer_id', 'Month'], as_index=False)['day']
    .max()
    .rename(columns={'day': 'last_active_day_prior_month', 'Month': 'Prior_Month'})
)

tx_prior_sorted = tx_prior.sort_values(['customer_id', 'Month', 'transaction_date'])
tx_prior_sorted['bal_after'] = (
    tx_prior_sorted['prior_month_open_balance'].fillna(0)
    + tx_prior_sorted.groupby(['customer_id', 'Month'])['amount'].cumsum()
)
tx_prior_sorted['rounded0'] = tx_prior_sorted['bal_after'].round(0) == 0
first_cross = (
    tx_prior_sorted.loc[tx_prior_sorted['rounded0']]
    .groupby(['customer_id', 'Month'], as_index=False)
    .first()[['customer_id', 'Month', 'day']]
    .rename(columns={'day': 'first_rounded0_day_prior', 'Month': 'Prior_Month'})
)

timing = (
    exit_keys[['customer_id', 'Month', 'Prior_Month']]
    .merge(last_active_prior, on=['customer_id', 'Prior_Month'], how='left')
    .merge(first_cross, on=['customer_id', 'Prior_Month'], how='left')
)

print('\n=== P-03 Intra-month timing (PRIOR month before first exit) ===')
print(
    'Note: first-exit months are inactive by definition — timing uses Prior_Month.'
)
print(
    f'Prior months with ≥1 active tx: '
    f'{timing["last_active_day_prior_month"].notna().sum():,} / {len(timing):,}'
)
print(
    f'Prior months with observed rounded-$0 cross day: '
    f'{timing["first_rounded0_day_prior"].notna().sum():,} / {len(timing):,}'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in [
    (axes[0], 'last_active_day_prior_month', 'Last active day — month before exit'),
    (axes[1], 'first_rounded0_day_prior', 'First rounded-$0 cross — month before exit'),
]:
    s = timing[col].dropna()
    if len(s):
        sns.histplot(s, bins=range(1, 33), discrete=True, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Day of month')
plt.tight_layout()
plt.show()

early_cross = timing['first_rounded0_day_prior'].dropna()
pct_by_day14 = float((early_cross <= 14).mean()) if len(early_cross) else float('nan')
print(
    f'Share of observed $0-crosses on day ≤14 (prior month): {pct_by_day14:.1%} '
    f'(n={len(early_cross):,})'
)
print(
    'Ops recommendation: score at month-start using end-of-prior-month features. '
    'If drains cluster in the first half of the pre-exit month, queue retention '
    'in the first days of the prediction month so ~14 days remain for Friction-Gate.'
)

# =============================================================================
# P-07 — Negative-balance months
# =============================================================================
neg = panel['Running_Balance'] < 0
neg_inactive = neg & (panel['Monthly_Active_Txn_Count'] == 0)
neg_rounded0 = neg & (panel['Running_Balance_Rounded'] == 0)
neg_not_label_shape = neg & ~(
    (panel['Running_Balance_Rounded'] == 0) & (panel['Monthly_Active_Txn_Count'] == 0)
)

print('\n=== P-07 Negative-balance months ===')
print(f'Customer-months: {len(panel):,}')
print(f'Running_Balance < 0: {int(neg.sum()):,} ({100 * neg.mean():.2f}%)')
print(f'  of which inactive: {int(neg_inactive.sum()):,}')
print(f'  of which rounded == 0: {int(neg_rounded0.sum()):,}')
print(
    f'  negative but NOT left-looking shape: {int(neg_not_label_shape.sum()):,} '
    f'(overdrafts / residuals — not labeled churn under rounded==0 + inactive)'
)
print(
    'Decision note: KEEP locked rule Running_Balance_Rounded == 0. '
    'Most negatives already round to 0; large-magnitude negatives that stay '
    'non-zero are not treated as churn. No ADR 004 change.'
)

# =============================================================================
# P-09 — Churn rate by age band + state (eligible rows)
# =============================================================================
eligible = panel.loc[panel['Is_Train_Eligible']].copy()
demo = customers_df[['customer_id', 'dob']].copy()
demo['dob'] = pd.to_datetime(demo['dob'], errors='coerce')
eligible = eligible.merge(demo, on='customer_id', how='left')
eligible['age'] = (AS_OF - eligible['dob']).dt.days / 365.25

age_bins = [0, 25, 35, 45, 55, 65, 120]
age_labels = ['<25', '25-34', '35-44', '45-54', '55-64', '65+']
eligible['age_band'] = pd.cut(
    eligible['age'], bins=age_bins, labels=age_labels, right=False
)

state_counts = eligible['state'].value_counts(dropna=False)
keep_states = set(state_counts[state_counts >= 5000].index)
eligible['state_group'] = eligible['state'].where(
    eligible['state'].isin(keep_states), other='OTHER/BAD'
)

age_ct = (
    eligible.groupby('age_band', observed=True)['Is_Churned']
    .agg(rows='size', positives='sum', rate='mean')
    .reset_index()
)
age_ct['rate_pct'] = (age_ct['rate'] * 100).map(lambda x: f'{x:.2f}%')

state_ct = (
    eligible.groupby('state_group', observed=True)['Is_Churned']
    .agg(rows='size', positives='sum', rate='mean')
    .reset_index()
    .sort_values('rows', ascending=False)
)
state_ct['rate_pct'] = (state_ct['rate'] * 100).map(lambda x: f'{x:.2f}%')

print('\n=== P-09 Churn by age band / state (Is_Train_Eligible) ===')
print('Age band × Is_Churned:')
display(age_ct[['age_band', 'rows', 'positives', 'rate_pct']])
print('State group × Is_Churned (states with <5k eligible rows → OTHER/BAD):')
display(state_ct[['state_group', 'rows', 'positives', 'rate_pct']].head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
age_rate = (
    eligible.groupby('age_band', observed=True)['Is_Churned'].mean().reindex(age_labels)
)
age_rate.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Positive rate by age band')
axes[0].set_ylabel('Is_Churned rate')
axes[0].tick_params(axis='x', rotation=0)

top_states = (
    eligible.groupby('state_group')['Is_Churned']
    .agg(rate='mean', rows='size')
    .sort_values('rows', ascending=False)
    .head(12)
)
top_states['rate'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Positive rate by state group (top 12 by volume)')
axes[1].set_ylabel('Is_Churned rate')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('\nP-02/P-03/P-07/P-09 complete. Label lock unchanged.')
